In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [16]:
# Load your data
data = pd.read_csv("data.csv")

# Data cleaning and preprocessing
def extract_numeric(value):
    if pd.isna(value):
        return np.nan
    values = [float(num) for num in value.replace(",", "").split() if num.replace('.', '', 1).isdigit()]
    return np.mean(values) if values else np.nan

In [22]:
def extract_numeric(value):
    if isinstance(value, (int, float)):  # If value is already numeric, return it
        return value
    if pd.isna(value):  # Handle NaN values
        return np.nan
    try:
        values = [float(num) for num in value.replace(",", "").split() if num.replace('.', '', 1).isdigit()]
        return np.mean(values) if values else np.nan
    except Exception as e:
        return np.nan  # Handle unexpected cases


In [23]:
data['temperature(process)'] = data['temperature(process)'].apply(extract_numeric)
data['pressure(process)'] = data['pressure(process)'].apply(extract_numeric)
if 'conversion' in data.columns:
    data['conversion'] = data['conversion'].apply(extract_numeric)
else:
    print("Warning: 'conversion' column not found in the dataset.")
data['product yield %'] = data['product yield %'].str.extract(r'(\d+\.?\d*)').astype(float)
cleaned_data = data.dropna(subset=['temperature(process)', 'pressure(process)', 'product yield %', 'conversion'])


In [24]:
vectorizer = TfidfVectorizer(max_features=10)
catalyst_features = vectorizer.fit_transform(cleaned_data['catalyst used and its specification']).toarray()


In [25]:
features = pd.DataFrame(catalyst_features, columns=[f'catalyst_feature_{i}' for i in range(catalyst_features.shape[1])])
features['temperature'] = cleaned_data['temperature(process)']
features['pressure'] = cleaned_data['pressure(process)']
features['conversion'] = cleaned_data['conversion']
target = cleaned_data['product yield %']
features = features.fillna(features.mean())

In [28]:
# Use all data for training
X_train = features
y_train = target

# Test the model with manual inputs later
model = RandomForestRegressor(random_state=42, n_estimators=100)
model.fit(X_train, y_train)


RandomForestRegressor(random_state=42)

In [31]:
custom_temperature = 250
custom_pressure = 5.0
custom_features = features.copy()
custom_features['temperature'] = custom_temperature
custom_features['pressure'] = custom_pressure
cleaned_data['predicted_yield_custom'] = model.predict(custom_features)


C:\Users\chand\AppData\Local\Temp\ipykernel_2116\1311362690.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_data['predicted_yield_custom'] = model.predict(custom_features)


In [30]:
best_catalyst_custom = cleaned_data.loc[cleaned_data['predicted_yield_custom'].idxmax()]
print("Best Catalyst for Custom Conditions:")
print("Catalyst:", best_catalyst_custom['catalyst used and its specification'])
print("Predicted Yield (%):", best_catalyst_custom['predicted_yield_custom'])
print("Custom Temperature (°C):", custom_temperature)
print("Custom Pressure (MPa):", custom_pressure)


Best Catalyst for Custom Conditions:
Catalyst: carbon-supported iron carbide, specifically Fe-Fe3C/C . These catalysts were synthesized using a high-temperature solid-phase method under a reducing atmosphere (10% H2 / 90% Ar) Structural Composition: Characterization techniques such as X-ray diffraction (XRD) and Raman spectroscopy revealed that the catalysts consist of graphitized and amorphous carbon encapsulating iron carbide nanocrystals and metal iron nanoclusters Performance in Lignin Conversion: The Fe-Fe3C/C catalysts demonstrated high selectivity for aromatic monomers, achieving an 86 wt% selectivity for guaiacol and its alkyl derivatives at a 48 wt% yield during optimal experimental conditions Mechanism of Action: The catalysts function by utilizing acid sites to adsorb and break C-O bonds, while metal Fe acts as a hydrogen source to promote hydrogenation processes. This dual functionality enhances the overall efficiency of lignin depolymerization 

Predicted Yield (%): 4.0
Cu